# 📊 Exploratory Data Analysis (EDA) - FloodGuard AI

This notebook covers the exploratory data analysis of the synthetic spatial-temporal flood risk dataset used in the **FloodGuard** application. We will visualize feature distributions, check correlations, and inspect how geospatial parameters like elevation and slope relate to simulated flood risk classes.

## Objectives:
1. Understand the dataset attributes (`latitude`, `longitude`, `elevation`, `slope`, `rainfall`, and lags).
2. Visualize the distribution of the target variable (`flood_risk`).
3. Perform correlation analysis to identify key predictors.
4. Analyze physical features (elevation, slope) and weather conditions (rainfall) against risk levels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries imported successfully!")

--- 
## 1. Load the Dataset

We load the generated dataset `flood_data.csv` from the `data/` directory.

In [ ]:
import os
dataset_path = "../data/flood_data.csv"

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
else:
    print(f"Error: Dataset not found at {dataset_path}. Run 'python src/data_generator.py' first.")

Let's look at the first few rows of data and inspect the column data types.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

--- 
## 2. Target Variable Distribution

The target column is `flood_risk` (0: Low Risk, 1: Medium Risk, 2: High Risk). Understanding the class balance is critical for model training.

In [ ]:
class_counts = df['flood_risk'].value_counts().sort_index()
class_percentages = df['flood_risk'].value_counts(normalize=True).sort_index() * 100

for c, count, pct in zip(class_counts.index, class_counts, class_percentages):
    print(f"Class {c}: {count} samples ({pct:.2f}%)")

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='flood_risk', palette="viridis")
plt.title("Distribution of Flood Risk Levels")
plt.xlabel("Risk Level (0: Low, 1: Medium, 2: High)")
plt.ylabel("Count")
plt.show()

--- 
## 3. Correlation Analysis

We calculate the Pearson correlation matrix to discover linear dependencies between geography, weather conditions, and risk level.

In [ ]:
corr = df.corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Pearson Feature Correlation Matrix")
plt.show()

### Insights:
- Look at the correlation coefficients relative to `flood_risk`.
- Notice the influence of current `rainfall` and preceding lag terms (`rainfall_lag_1`, `rainfall_lag_2`) which model soil saturation parameters.

--- 
## 4. Feature Analysis against Flood Risk

Let's look at how individual parameters like elevation and slope impact the severity of risk.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Elevation by Risk Level
sns.boxplot(ax=axes[0], data=df, x='flood_risk', y='elevation', palette="Set2")
axes[0].set_title("Elevation distribution by Flood Risk Level")
axes[0].set_xlabel("Flood Risk Level")
axes[0].set_ylabel("Elevation (meters)")

# Slope by Risk Level
sns.boxplot(ax=axes[1], data=df, x='flood_risk', y='slope', palette="Set2")
axes[1].set_title("Slope distribution by Flood Risk Level")
axes[1].set_xlabel("Flood Risk Level")
axes[1].set_ylabel("Slope (degrees)")

plt.tight_layout()
plt.show()

### Rainfall vs. Risk Levels
Let's see how much rainfall is typically required to transition between risk levels.

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x='flood_risk', y='rainfall', palette="muted")
plt.title("Rainfall Distribution by Risk Class")
plt.xlabel("Flood Risk Class")
plt.ylabel("Rainfall (mm)")
plt.show()

--- 
## Conclusions for Model Design
1. **Geospatial Variables:** Low elevation and low slope areas show higher flood risk susceptibility due to gravity-driven pooling.
2. **Weather Features:** High rainfall is the primary trigger, but preceding lag terms are critical indicators of pre-existing soil saturation levels.
3. **Algorithm:** Because the interaction of low elevation, low slope, and cumulative rain is highly non-linear, tree-based models like **XGBoost** are ideal for modeling this environment.